# NSL-KDD: Data Engineering for Network Intrusion Classification

Course project: 204426, term 1/69. A data-quality and feature-preparation pipeline with a downstream five-class Random Forest classifier.

The four setups below are compared descriptively on KDDTest+. The historical notebook did **not** select its setup using an executed train-only CV experiment. See `../docs/reproducibility.md` for limitations and publication edits. Historical rounded metrics are preserved separately in `../results/`.


In [ ]:
import pandas as pd, numpy as np, urllib.request
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, f1_score, recall_score)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
SEED = 42
N_EST = 100   # จำนวนต้นไม้ (ลดเวลารัน ablation)
from pathlib import Path
import os
from IPython.display import display
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = Path(os.environ.get("NSLKDD_DATA_DIR", str(REPO_ROOT / "data")))
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = REPO_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Pipeline Diagram (Input → Process → Output)

In [ ]:
steps = [
    ("INPUT\nNSL-KDD\nKDDTrain+ / KDDTest+", "#dbeafe"),
    ("1 Extraction\nโหลด + ใส่ schema\n(43 คอลัมน์)", "#e0e7ff"),
    ("2 Quality Check\nMissing / Duplicate\n/ Constant / Skew", "#fef9c3"),
    ("3 Cleaning\nลบ duplicate\nลบ constant", "#fde68a"),
    ("4 Feature Selection\ncorrelation > 0.95", "#fed7aa"),
    ("5 Feature Eng.\n(ทดลอง + ablation)", "#fecaca"),
    ("6 Preprocessing\nOne-hot (+ Scaler\nเฉพาะ LR)", "#e9d5ff"),
    ("7 Validation\nassert หลังทุกขั้น", "#bbf7d0"),
    ("OUTPUT\nClean data → Model\n→ Evaluation", "#dbeafe"),
]
fig, ax = plt.subplots(figsize=(16, 3.2))
ax.set_xlim(0, len(steps)*2.2); ax.set_ylim(0, 3); ax.axis("off")
for i, (t, c) in enumerate(steps):
    x = i*2.2 + 0.1
    ax.add_patch(FancyBboxPatch((x, 0.6), 1.9, 1.8, boxstyle="round,pad=0.05", fc=c, ec="#334155"))
    ax.text(x+0.95, 1.5, t, ha="center", va="center", fontsize=8)
    if i < len(steps)-1:
        ax.add_patch(FancyArrowPatch((x+1.95, 1.5), (x+2.25, 1.5), arrowstyle="->", mutation_scale=14, color="#334155"))
ax.set_title("Data Engineering Pipeline: NSL-KDD attack classification", fontsize=11)
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "pipeline_diagram.png", dpi=150); plt.show()

## 2. Data Extraction
โหลดข้อมูล + กำหนด schema (ชื่อคอลัมน์) และตรวจว่าจำนวนคอลัมน์ตรงกับที่คาดไว้

In [ ]:
BASE = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
urllib.request.urlretrieve(BASE + "KDDTrain%2B.txt", DATA_DIR / "KDDTrain.txt")
urllib.request.urlretrieve(BASE + "KDDTest%2B.txt", DATA_DIR / "KDDTest.txt")

cols = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent",
"hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations",
"num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count",
"serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
"dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
"dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

train = pd.read_csv(DATA_DIR / "KDDTrain.txt", names=cols)
test  = pd.read_csv(DATA_DIR / "KDDTest.txt",  names=cols)

# Validation ของขั้น extraction
assert train.shape[1] == test.shape[1] == 43, "จำนวนคอลัมน์ไม่ตรง schema"
assert list(train.columns) == list(test.columns)
print("Train:", train.shape, "| Test:", test.shape)
train.head()

## 3. จัดกลุ่ม attack เป็น 5 หมวด (target)

In [ ]:
dos   = "back land neptune pod smurf teardrop apache2 udpstorm processtable worm mailbomb".split()
probe = "ipsweep nmap portsweep satan mscan saint".split()
r2l   = "ftp_write guess_passwd imap multihop phf spy warezclient warezmaster xlock xsnoop snmpguess snmpgetattack httptunnel sendmail named".split()
u2r   = "buffer_overflow loadmodule perl rootkit ps sqlattack xterm".split()

def to_category(l):
    if l == "normal": return "Normal"
    if l in dos:   return "DoS"
    if l in probe: return "Probe"
    if l in r2l:   return "R2L"
    if l in u2r:   return "U2R"
    raise ValueError(f"label ไม่รู้จัก: {l}")

for d in (train, test):
    d["Attack Type"] = d["label"].map(to_category)

LABELS = ["Normal","DoS","Probe","R2L","U2R"]
counts = pd.DataFrame({"train": train["Attack Type"].value_counts(),
                       "test":  test["Attack Type"].value_counts()}).loc[LABELS]
print(counts)

unseen = sorted(set(test["label"]) - set(train["label"]))
print("\nattack subtype ที่มีเฉพาะใน test:", unseen)
print("จำนวนแถวของ subtype เหล่านี้ใน test:", int(test["label"].isin(unseen).sum()), "/", len(test))
counts.plot(kind="bar", logy=True, figsize=(7,4), title="Class distribution (log scale)")
plt.ylabel("count"); plt.show()

## 4. Data Engineering
การตัดสินใจทุกอย่างในขั้นนี้ใช้ **train set เท่านั้น** (ไม่ดู test) เพื่อไม่ให้เกิด data leakage

### 4.0 ตัวช่วย: Validation + Step Log
ทุกขั้นตอนจะเรียก `validate()` และบันทึกจำนวนแถว/feature ลงตาราง step log

In [ ]:
CAT = ["protocol_type", "service", "flag"]
step_log = []

def validate(step, Xtr, ytr, Xte, yte):
    """ตรวจความถูกต้องของข้อมูลหลังแต่ละขั้นตอน แล้วบันทึกลง step_log"""
    assert Xtr.isna().sum().sum() == 0, f"[{step}] train มี missing"
    assert Xte.isna().sum().sum() == 0, f"[{step}] test มี missing"
    assert len(Xtr) == len(ytr), f"[{step}] len(X_train) != len(y_train)"
    assert len(Xte) == len(yte), f"[{step}] len(X_test) != len(y_test)"
    assert list(Xtr.columns) == list(Xte.columns), f"[{step}] schema train/test ไม่ตรงกัน"
    assert set(CAT).issubset(Xtr.columns), f"[{step}] คอลัมน์ categorical หาย"
    assert set(ytr.unique()).issubset(LABELS), f"[{step}] y_train มี label แปลก"
    step_log.append(dict(step=step, train_rows=len(Xtr), test_rows=len(Xte), n_features=Xtr.shape[1]))
    print(f"OK  {step:<32} train={Xtr.shape}  test={Xte.shape}")

### 4.1 ตรวจคุณภาพข้อมูล (Data Profiling)

In [ ]:
DROP = ["label", "difficulty", "Attack Type"]
X_train_raw, y_train = train.drop(columns=DROP), train["Attack Type"]
X_test_raw,  y_test  = test.drop(columns=DROP),  test["Attack Type"]
validate("0 raw", X_train_raw, y_train, X_test_raw, y_test)

n_missing = int(X_train_raw.isna().sum().sum())
n_dup_train = int(X_train_raw.duplicated().sum())
n_dup_test  = int(X_test_raw.duplicated().sum())
const_cols = [c for c in X_train_raw.columns if X_train_raw[c].nunique() == 1]
print("\nMissing (train):", n_missing)
print("Duplicate  train:", n_dup_train, "| test:", n_dup_test)
print("Constant columns:", const_cols)

num_raw = X_train_raw.select_dtypes(include="number")
print("\nSkewness สูงสุด 5 อันดับ:")
print(num_raw.skew().abs().sort_values(ascending=False).head(5).round(1))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top5_skew = [
    "is_host_login",
    "dst_bytes",
    "num_compromised",
    "num_root",
    "src_bytes"
]

fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for ax, col in zip(axes, top5_skew):
    sns.boxplot(y=X_train_raw[col], ax=ax)
    ax.set_title(col)
    ax.set_ylabel("Value")

plt.suptitle("Box Plot of Top 5 Most Skewed Features", y=1.05)
plt.tight_layout()
plt.show()

### 4.2 ลบ duplicate และ constant column

In [ ]:
# ลบ duplicate เฉพาะ train (เก็บ test ตามต้นฉบับ เพื่อเทียบกับงานอื่นได้)
keep = ~X_train_raw.duplicated()
X_train_de, y_train_de = X_train_raw[keep], y_train[keep]
X_train_de = X_train_de.drop(columns=const_cols)
X_test_de  = X_test_raw.drop(columns=const_cols)
validate("1 dedup + drop constant", X_train_de, y_train_de, X_test_de, y_test)

### 4.3 ลบ feature ที่ correlate สูง (> 0.95)
feature ซ้ำซ้อนไม่ช่วยโมเดล และทำให้ overfit ง่ายขึ้น คำนวณจาก train เท่านั้น

In [ ]:
corr = X_train_de.select_dtypes(include="number").corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_drop = [c for c in upper.columns if (upper[c] > 0.95).any()]
print("Dropping correlated features:", corr_drop)

# แสดงคู่ที่ correlate สูง เพื่อเป็นหลักฐานว่าทำไมถึงลบ
pairs = upper.stack().reset_index()
pairs.columns = ["feature_A", "feature_B", "abs_corr"]
pairs = pairs[pairs["abs_corr"] > 0.95].sort_values("abs_corr", ascending=False)
display(pairs.round(3).reset_index(drop=True))

X_train_sel = X_train_de.drop(columns=corr_drop)
X_test_sel  = X_test_de.drop(columns=corr_drop)
validate("2 drop correlated (>0.95)", X_train_sel, y_train_de, X_test_sel, y_test)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 12))
sns.heatmap(corr, annot=False, cmap='viridis', fmt=".2f", linewidths=.5)
plt.title('Heatmap of Absolute Correlation Matrix (after deduplication and constant feature removal)')
plt.show()

### 4.4 Data Quality Report
สรุปปัญหาที่พบในข้อมูลจริง และ action ที่เลือกทำ (ตัวเลขคำนวณจากโค้ดด้านบน)

In [ ]:
min_class = y_train.value_counts().idxmin()
quality_report = pd.DataFrame([
    ["Missing value",          f"{n_missing}",                                  "ไม่ต้องทำอะไร"],
    ["Duplicate rows",         f"train {n_dup_train} | test {n_dup_test}",      "ลบจาก train (เก็บ test ตามต้นฉบับ)"],
    ["Constant feature",       f"{len(const_cols)}: {const_cols}",              "ลบ (ไม่มีข้อมูล)"],
    ["Highly correlated (>0.95)", f"{len(corr_drop)}: {corr_drop}",             "ลบ (ซ้ำซ้อน)"],
    ["Categorical feature",    f"{len(CAT)}: {CAT}",                            "One-hot encoding (handle_unknown=ignore)"],
    ["Skewed numeric",         "src_bytes, dst_bytes, ... (skew > 190)",        "ทดลอง log-transform ใน feature engineering"],
    ["Class imbalance",        f"{min_class}={int(y_train.value_counts().min())} vs Normal={int(y_train.value_counts()['Normal'])}",
                                                                                "class_weight='balanced' + รายงาน Macro F1"],
    ["Unseen attack in test",  f"{len(unseen)} subtypes ({int(test['label'].isin(unseen).sum())} แถว)",
                                                                                "รายงานเป็นข้อจำกัด + error analysis"],
], columns=["Data Quality Check", "Result", "Action"])
display(quality_report)

### 4.5 Feature Engineering (ทดลอง)
สร้าง ratio, log-transform, รวม error rate, จัดกลุ่ม service ที่หายาก แล้ววัดผลใน ablation ด้านล่าง — **ไม่ได้ใช้เพียงเพราะคาดว่าจะช่วย**

In [ ]:
top_services = set(X_train_sel["service"].value_counts().head(15).index)  # จาก train เท่านั้น

def add_features(d):
    d = d.copy()
    d["bytes_ratio"]    = d["src_bytes"] / (d["dst_bytes"] + 1)
    d["src_bytes_log"]  = np.log1p(d["src_bytes"])
    d["dst_bytes_log"]  = np.log1p(d["dst_bytes"])
    d["duration_log"]   = np.log1p(d["duration"])
    d["err_rate_mean"]  = d[["serror_rate","rerror_rate"]].mean(axis=1)
    d["srv_ratio"]      = d["srv_count"] / (d["count"] + 1)
    d["host_srv_ratio"] = d["dst_host_srv_count"] / (d["dst_host_count"] + 1)
    sus = [c for c in ["hot","num_failed_logins","num_compromised","root_shell",
                       "su_attempted","num_file_creations","num_shells","num_access_files"] if c in d.columns]
    d["suspicious_cnt"] = d[sus].sum(axis=1)
    d["service"] = d["service"].where(d["service"].isin(top_services), "other")
    return d

X_train_fe, X_test_fe = add_features(X_train_sel), add_features(X_test_sel)
validate("3 + engineered features", X_train_fe, y_train_de, X_test_fe, y_test)

### 4.6 Step Log (จำนวนแถว/feature ก่อน-หลังแต่ละขั้น)

In [ ]:
step_df = pd.DataFrame(step_log).set_index("step")
step_df["features_change"] = step_df["n_features"].diff().fillna(0).astype(int)
step_df["train_rows_change"] = step_df["train_rows"].diff().fillna(0).astype(int)
display(step_df)

## 5. Preprocessing + Model
- One-hot สำหรับ categorical
- **StandardScaler ใช้เฉพาะ Logistic Regression** (ไวต่อ scale) — Random Forest ไม่จำเป็นต้อง scale จึงเก็บ numeric ไว้ scale เดิม (มีการทดลองเทียบใน section 7 เพื่อยืนยัน)
- Random Forest แบบจำกัดความซับซ้อน (`max_depth=8`, `min_samples_leaf=20`) + `class_weight='balanced'` เพื่อลด overfit และ imbalance

In [ ]:
def make_model(X, kind="rf", scale=None):
    """kind: 'dummy' | 'lr' | 'rf'.  scale=None -> lr ใช้ scaler, rf/dummy ไม่ใช้"""
    if scale is None:
        scale = (kind == "lr")
    num = [c for c in X.columns if c not in CAT]
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("num", StandardScaler() if scale else "passthrough", num),
    ])
    if kind == "dummy":
        clf = DummyClassifier(strategy="most_frequent")
    elif kind == "lr":
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    else:
        clf = RandomForestClassifier(n_estimators=N_EST, max_depth=8, min_samples_leaf=20,
                                     class_weight="balanced", n_jobs=-1, random_state=SEED)
    return Pipeline([("pre", pre), ("clf", clf)])

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def evaluate_cv_diagnostic(name, Xtr, ytr, Xte, yte, kind="rf", scale=None):
    m = make_model(Xtr, kind, scale)
    cv = cross_val_score(m, Xtr, ytr, cv=skf, scoring="f1_macro", n_jobs=1).mean()   # เลือกค่าด้วย train
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    rec = recall_score(yte, p, labels=LABELS, average=None, zero_division=0)
    return dict(setup=name, n_features=Xtr.shape[1],
                cv_macro_f1=cv,
                train_acc=accuracy_score(ytr, m.predict(Xtr)),
                test_acc=accuracy_score(yte, p),
                test_macro_f1=f1_score(yte, p, average="macro", zero_division=0),
                recall_R2L=rec[3], recall_U2R=rec[4])

## 6. Descriptive Test-Set Ablation

Compare four data-engineering setups using the same Random Forest configuration. These are **test-set** metrics, not CV scores. Setup 2 is retained for the detailed report because it had the largest recorded test macro F1. A new, untouched evaluation set would be required after such a selection.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_test_metrics(name, Xtr, ytr, Xte, yte):

    model = make_model(Xtr, kind="rf")
    model.fit(Xtr, ytr)

    y_pred = model.predict(Xte)

    return {
        "setup": name,
        "Accuracy": accuracy_score(yte, y_pred),
        "Precision": precision_score(
            yte, y_pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            yte, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1-score": f1_score(
            yte, y_pred,
            average="macro",
            zero_division=0
        )
    }


setups = {

    "0 raw (no engineering)":      (X_train_raw, y_train,    X_test_raw),

    "1 + dedup + drop constant":   (X_train_de,  y_train_de, X_test_de),

    "2 + drop correlated (>0.95)": (X_train_sel, y_train_de, X_test_sel),

    "3 + engineered features":     (X_train_fe,  y_train_de, X_test_fe),

}

results = [
    evaluate_test_metrics(n, Xtr, ytr, Xte, y_test)
    for n, (Xtr, ytr, Xte) in setups.items()
]

ablation = (
    pd.DataFrame(results)
    .set_index("setup")
    .round(3)
)

display(ablation)

### Interpretation and limitations

In the supplied saved output, setup 2 recorded accuracy **0.791** and macro F1 **0.680**. Engineered features recorded macro F1 **0.627**; a lower test score alone does not establish overfitting.

The CV diagnostic helper above is not called by this ablation. Training-set-derived feature filtering and engineering are outside that helper's folds, so it is not a fully fold-isolated model-selection pipeline. Claims about historical CV selection were removed because they did not match the active code.

The next cells retain setup 2's confusion matrix and per-class classification report. These are benchmark results, not evidence of production network protection.


In [ ]:
print("Generating Confusion Matrix for '2 + drop correlated (>0.95)' setup...")

specific_setup_name = "2 + drop correlated (>0.95)"
X_train_specific, y_train_specific, X_test_specific = setups[specific_setup_name]

# Retrain the model for this specific setup to get the predictions
m = make_model(X_train_specific, kind='rf')
m.fit(X_train_specific, y_train_specific)
p_specific = m.predict(X_test_specific)

# Calculate and display the confusion matrix
cm = confusion_matrix(y_test, p_specific, labels=LABELS)
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
fig, ax = plt.subplots(figsize=(8, 8))
cmd.plot(ax=ax, cmap='Blues')
plt.title(f"Confusion Matrix for '{specific_setup_name}'")
plt.show()

In [ ]:
print(f"\nClassification Report for '{specific_setup_name}':")
print(classification_report(y_test, p_specific, labels=LABELS, zero_division=0))